# 03 - Collecter depuis une base de données 

## 1 · Depuis une base MySQL

In [13]:
# ÉTAPE 1 : connexion entre Python et MySQL
# Création du fichier .env avec identifiant et mot de passe MySQL.
# Le mot de passe est tapé dans une case masquée : il n'apparaît jamais dans le notebook.
%pip install pymysql python-dotenv sqlalchemy
from getpass import getpass

utilisateur = input("Utilisateur MySQL (souvent root) : ")
mot_de_passe = getpass("Mot de passe MySQL : ")
with open("../.env", "w", encoding="utf-8") as fichier:
    fichier.write(f"MYSQL_UTILISATEUR={utilisateur}\nMYSQL_MOT_DE_PASSE={mot_de_passe}\n")
print("Fichier .env créé.")

Note: you may need to restart the kernel to use updated packages.


Utilisateur MySQL (souvent root) :  root
Mot de passe MySQL :  ········


Fichier .env créé.


In [18]:
# ÉTAPE 2 : me connecter à ma base MySQL
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, URL

load_dotenv("../.env")
url = URL.create("mysql+pymysql", username=os.getenv("MYSQL_UTILISATEUR"), password=os.getenv("MYSQL_MOT_DE_PASSE"),
                 host="localhost", port=3306, database="airbus_supplychain")
moteur = create_engine(url)

In [19]:
# ÉTAPE 3 : envoie de mes 3 fichiers dans MySQL, un par un

# TABLE 1 : parts_master 
parts_master = pd.read_csv("../data/raw/parts_master.csv")      
parts_master.to_sql("parts_master", moteur, if_exists="replace", index=False)

print("parts_master :", len(parts_master), "lignes envoyées")    # je dois obtenir 300   

# TABLE 2 : purchase_orders 
purchase_orders = pd.read_csv("../data/raw/purchase_orders.csv")
purchase_orders.to_sql("purchase_orders", moteur, if_exists="replace", index=False)
print("purchase_orders :", len(purchase_orders), "lignes envoyées")   # je dois obtenir 29666

# TABLE 3 : supply_chain_history (280 800 lignes)
supply_chain_history = pd.read_csv("../data/raw/supply_chain_history.csv")
supply_chain_history.to_sql("supply_chain_history", moteur, if_exists="replace", index=False, chunksize=10000)
print("supply_chain_history :", len(supply_chain_history), "lignes envoyées")    # je dois obtenir 280800

parts_master : 300 lignes envoyées
purchase_orders : 29666 lignes envoyées
supply_chain_history : 280800 lignes envoyées


In [21]:
# Je vérifie la taille de mes tables dans SQL
print(pd.read_sql("SELECT COUNT(*) AS nb_lignes FROM parts_master", moteur))
print(pd.read_sql("SELECT COUNT(*) AS nb_lignes FROM purchase_orders", moteur))
print(pd.read_sql("SELECT COUNT(*) AS nb_lignes FROM supply_chain_history", moteur))

   nb_lignes
0        300
   nb_lignes
0      29666
   nb_lignes
0     280800


In [ ]:
# Ta question métier : <écris-la ici>
requete = """
SELECT <colonnes utiles>
FROM <table>
JOIN <autre table> ON <clé commune>
WHERE <filtre>
GROUP BY <regroupement>
"""
resultat = pd.read_sql(requete, moteur)
resultat.to_csv("../data/raw/<nom_du_fichier>.csv", index=False)
resultat.head()